# Day 1: Multi-Agent Incident Triage & Orchestration

## Advanced Multi-Agent AI Systems with LangChain & LangGraph
### For Support & Meta Engineers

---

## 🎯 What You'll Learn

In this hands-on lab, you will:
1. Build multi-agent systems using **LangChain** and **LangGraph**
2. Orchestrate complex workflows with **StateGraph**
3. Implement structured outputs with **Pydantic schemas**
4. Add production **guardrails** (max steps, loop detection, tool allowlists)
5. Enable **observability** with JSONL trace logging
6. Handle **failure modes** and recovery patterns

## 📊 What You'll Build

A **3-agent incident triage system** that:
- **Classifies** incidents (severity P0-P4, category)
- **Deduplicates** similar incidents (alert storm detection)
- **Routes** to appropriate teams (SLA-aware)

All orchestrated via **LangGraph StateGraph** with full observability.

---

## Part 1: Setup & Environment

### 1.1 Import Dependencies

In [ ]:
import sys
import os
import json
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✓ Project root: {project_root}")
print(f"✓ Python version: {sys.version.split()[0]}")

### 1.2 Initialize LLM (MockLLM or OpenAI)

**IMPORTANT**: This notebook runs **without API keys** using MockLLM.

To use OpenAI:
1. Create `.env` file in project root
2. Add: `OPENAI_API_KEY=sk-your-key-here`
3. Restart kernel

In [ ]:
from src.llm_factory import get_llm, print_llm_stats

llm = get_llm(force_mock=False, deterministic=True, verbose=True)

print("\n✓ LLM initialized and ready")

### 1.3 Load Sample Data

In [ ]:
incidents = []
with open(project_root / 'data' / 'incidents_large_small.jsonl', 'r') as f:
    for line in f:
        incidents.append(json.loads(line.strip()))

with open(project_root / 'data' / 'kb_policies.json', 'r') as f:
    policies = json.load(f)

with open(project_root / 'data' / 'sample_logs.txt', 'r') as f:
    sample_logs = f.read()

print(f"✓ Loaded {len(incidents)} incidents")
print(f"✓ Loaded policies: {list(policies.keys())}")
print(f"✓ Loaded {len(sample_logs.splitlines())} log lines")

print("\nSample Incidents:")
for i, inc in enumerate(incidents[:3], 1):
    print(f"{i}. {inc['id']}: {inc['title']} [{inc['severity']}]")

---

## Part 2: Pydantic Schemas for Structured Outputs

### 2.1 Understanding Structured Outputs

**Why Pydantic schemas?**
- ✅ Type safety and validation
- ✅ Automatic error detection
- ✅ Clear contracts between agents
- ✅ Production-ready data models

In [ ]:
from src.schemas import (
    ClassificationResult,
    DeduplicationResult,
    RoutingDecision,
    SeverityLevel,
    IncidentCategory
)

print("Available Schemas:")
print(f"  - ClassificationResult: {ClassificationResult.__fields__.keys()}")
print(f"  - DeduplicationResult: {DeduplicationResult.__fields__.keys()}")
print(f"  - RoutingDecision: {RoutingDecision.__fields__.keys()}")

print("\nSeverity Levels:", [s.value for s in SeverityLevel])
print("Categories:", [c.value for c in IncidentCategory])

### 2.2 Schema Validation Example

In [ ]:
try:
    valid_classification = ClassificationResult(
        severity=SeverityLevel.P0,
        category=IncidentCategory.DATABASE,
        confidence=0.95,
        reasoning="Database connection pool exhausted",
        requires_escalation=True
    )
    print("✓ Valid classification created")
    print(json.dumps(valid_classification.dict(), indent=2))
except Exception as e:
    print(f"✗ Validation error: {e}")

print("\n" + "="*60)
print("Testing Invalid Data:")
print("="*60)

try:
    invalid_classification = ClassificationResult(
        severity="P99",
        category="InvalidCategory",
        confidence=1.5,
        reasoning="Test"
    )
except Exception as e:
    print(f"✓ Caught validation error (expected): {type(e).__name__}")
    print(f"  Message: {str(e)[:100]}...")

---

## Part 3: LangGraph StateGraph Workflow

### 3.1 Understanding StateGraph

**LangGraph StateGraph** is a state machine for orchestrating agents:
- **Nodes**: Individual agent operations
- **Edges**: Transitions between nodes
- **State**: Shared data passed between nodes
- **Conditional edges**: Dynamic routing based on state

```
┌─────────────┐
│  Classify   │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Guardrails  │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Deduplicate │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│    Route    │
└─────────────┘
```

### 3.2 Initialize Observability

In [ ]:
from src.observability import TraceLogger, create_trace_viewer

logger = TraceLogger(log_file="traces_day1.jsonl")
logger.clear()

print("✓ Trace logger initialized")
print(f"  Log file: traces_day1.jsonl")

### 3.3 Create Triage Workflow

In [ ]:
from src.workflow_day1 import create_triage_workflow

triage_workflow = create_triage_workflow(
    llm=llm,
    logger=logger,
    max_steps=10,
    incident_db=incidents,
    policies=policies
)

print("✓ Triage workflow created")
print(f"  Max steps: 10")
print(f"  Incident DB size: {len(incidents)}")

### 3.4 Run Single Incident Through Workflow

In [ ]:
test_incident = incidents[0]

print("\n" + "="*60)
print(f"Processing: {test_incident['id']}")
print(f"Title: {test_incident['title']}")
print(f"Description: {test_incident['description'][:100]}...")
print("="*60 + "\n")

result = triage_workflow.run(test_incident)

print("\n" + "="*60)
print("Workflow Result:")
print("="*60)
print(f"Status: {result['status']}")
print(f"Trace ID: {result['trace_id']}")
print(f"Steps: {result['step_count']}")
print(f"Errors: {len(result.get('errors', []))}")

### 3.5 Examine Classification Result

In [ ]:
classification = result.get('classification', {})

print("\n📊 CLASSIFICATION RESULT")
print("="*60)
if classification:
    print(json.dumps(classification, indent=2))
else:
    print("No classification result available")

### 3.6 Examine Deduplication Result

In [ ]:
deduplication = result.get('deduplication', {})

print("\n📊 DEDUPLICATION RESULT")
print("="*60)
if deduplication:
    print(json.dumps(deduplication, indent=2))
else:
    print("No deduplication result available")

### 3.7 Examine Routing Result

In [ ]:
routing = result.get('routing', {})

print("\n📊 ROUTING RESULT")
print("="*60)
if routing:
    print(json.dumps(routing, indent=2))
else:
    print("No routing result available")

---

## Part 4: Observability & Trace Viewing

### 4.1 View Trace Summary

In [ ]:
trace_summary = logger.get_trace_summary(result['trace_id'])

print("\n📊 TRACE SUMMARY")
print("="*60)
print(json.dumps(trace_summary, indent=2))

### 4.2 View All Traces as DataFrame

In [ ]:
traces_df = create_trace_viewer(logger)

print("\n📊 ALL TRACES")
print("="*60)
if not traces_df.empty:
    display(traces_df)
else:
    print("No traces available")

### 4.3 Detailed Trace Visualization

In [ ]:
from src.observability import format_trace_for_display

trace_display = format_trace_for_display(logger, result['trace_id'])
print(trace_display)

---

## Part 5: Guardrails in Action

### 5.1 Max Steps Guardrail

Test what happens when max steps is exceeded:

In [ ]:
print("\n" + "="*60)
print("Testing Max Steps Guardrail")
print("="*60 + "\n")

limited_workflow = create_triage_workflow(
    llm=llm,
    logger=logger,
    max_steps=2,
    incident_db=incidents,
    policies=policies
)

result_limited = limited_workflow.run(incidents[1])

print(f"Status: {result_limited['status']}")
print(f"Steps executed: {result_limited['step_count']}")
print(f"Errors: {result_limited.get('errors', [])}")

### 5.2 Error Handling

Test handling of invalid JSON responses:

In [ ]:
print("\n" + "="*60)
print("Testing Error Handling")
print("="*60 + "\n")

invalid_incident = {
    "id": "TEST-001",
    "title": "",
    "description": "",
    "severity": "unknown"
}

result_error = triage_workflow.run(invalid_incident)

print(f"Status: {result_error['status']}")
print(f"Errors encountered: {len(result_error.get('errors', []))}")
if result_error.get('errors'):
    print("\nError details:")
    for i, error in enumerate(result_error['errors'][:3], 1):
        print(f"  {i}. {error[:100]}...")

---

## Part 6: Batch Processing

### 6.1 Process Multiple Incidents

In [ ]:
print("\n" + "="*60)
print("Batch Processing Incidents")
print("="*60 + "\n")

batch_results = []

for i, incident in enumerate(incidents[:5], 1):
    print(f"{i}. Processing {incident['id']}: {incident['title'][:40]}...")
    
    result = triage_workflow.run(incident)
    
    batch_results.append({
        'incident_id': incident['id'],
        'status': result['status'],
        'step_count': result['step_count'],
        'has_errors': len(result.get('errors', [])) > 0,
        'classification': result.get('classification', {}).get('severity'),
        'routing': result.get('routing', {}).get('assigned_team')
    })
    
    print(f"   ✓ {result['status']} - {result['step_count']} steps\n")

print("\n" + "="*60)
print("Batch Summary")
print("="*60)

batch_df = pd.DataFrame(batch_results)
display(batch_df)

### 6.2 Analyze Batch Results

In [ ]:
print("\n📊 BATCH ANALYSIS")
print("="*60)

success_rate = (batch_df['status'] == 'completed').sum() / len(batch_df)
avg_steps = batch_df['step_count'].mean()
error_rate = batch_df['has_errors'].sum() / len(batch_df)

print(f"Total Incidents: {len(batch_df)}")
print(f"Success Rate: {success_rate:.1%}")
print(f"Average Steps: {avg_steps:.1f}")
print(f"Error Rate: {error_rate:.1%}")

print("\nClassification Distribution:")
print(batch_df['classification'].value_counts())

print("\nRouting Distribution:")
print(batch_df['routing'].value_counts())

---

## Part 7: Alert Storm Detection

### 7.1 Detect Duplicate Incidents

In [ ]:
from src.clustering import detect_alert_storm, IncidentClusterer

print("\n" + "="*60)
print("Alert Storm Detection")
print("="*60 + "\n")

alert_storm_result = detect_alert_storm(
    incidents=incidents[:15],
    time_window_minutes=60,
    min_incidents=3
)

print(json.dumps(alert_storm_result, indent=2))

### 7.2 Cluster Similar Incidents

In [ ]:
clusterer = IncidentClusterer(similarity_threshold=0.70)
clusters = clusterer.cluster_incidents(incidents[:10], min_cluster_size=2)

print("\n📊 INCIDENT CLUSTERS")
print("="*60)
print(f"Total clusters found: {len(clusters)}\n")

for cluster_id, cluster_incidents in clusters.items():
    print(f"\n{cluster_id}:")
    print(f"  Size: {len(cluster_incidents)}")
    
    summary = clusterer.get_cluster_summary(cluster_incidents)
    print(f"  Common Severity: {summary['most_common_severity']}")
    print(f"  Common Category: {summary['most_common_category']}")
    print(f"  Incidents: {summary['incident_ids']}")

---

## Part 8: LangChain Tools

### 8.1 Understanding Tools

**LangChain Tools** provide structured interfaces for:
- **Read operations**: Search, query, lookup (safe)
- **Write operations**: Update, escalate (require approval)

**Tool Allowlists** are a key guardrail for production.

In [ ]:
from src.tools import get_read_only_tools, get_default_tools

read_only_tools = get_read_only_tools(
    incident_db=incidents,
    policies=policies,
    log_data=sample_logs
)

print("\n📦 READ-ONLY TOOLS")
print("="*60)
for tool in read_only_tools:
    print(f"  - {tool.name}: {tool.description[:60]}...")

all_tools = get_default_tools(
    incident_db=incidents,
    policies=policies,
    log_data=sample_logs,
    allow_writes=True
)

print("\n📦 ALL TOOLS (including writes)")
print("="*60)
for tool in all_tools:
    print(f"  - {tool.name}: {tool.description[:60]}...")

### 8.2 Test Tool Execution

In [ ]:
search_tool = read_only_tools[0]

print("\n🔧 Testing Tool: search_incidents")
print("="*60)

result = search_tool.run({"query": "database", "limit": 3})
print(json.dumps(json.loads(result), indent=2))

---

## Part 9: Checkpoint & Review

### What We've Learned

✅ **LangGraph StateGraph** for workflow orchestration  
✅ **Pydantic schemas** for structured outputs  
✅ **Guardrails**: max steps, error handling  
✅ **Observability**: JSONL traces, trace IDs  
✅ **LangChain Tools** with read/write separation  
✅ **Alert storm detection** and clustering  

### LLM Usage Statistics

In [ ]:
print_llm_stats(llm)

### Trace Statistics

In [ ]:
all_traces = create_trace_viewer(logger)

if not all_traces.empty:
    print("\n📊 TRACE STATISTICS")
    print("="*60)
    print(f"Total Traces: {len(all_traces)}")
    print(f"Completed: {(all_traces['status'] == 'completed').sum()}")
    print(f"Failed: {(all_traces['status'] == 'failed').sum()}")
    print(f"\nAverage Duration: {all_traces['total_duration_ms'].mean():.2f}ms")
    print(f"Total Tool Calls: {all_traces['tool_calls_count'].sum()}")
    print(f"Total Errors: {all_traces['errors_count'].sum()}")
    print(f"Total Violations: {all_traces['violations_count'].sum()}")
else:
    print("No trace data available")

---

## 🎉 Day 1 Complete!

You've successfully built a **production-ready incident triage system** using:
- ✅ LangChain & LangGraph
- ✅ Structured outputs with Pydantic
- ✅ Production guardrails
- ✅ Full observability

### Next Steps

**Day 2** will cover:
- 5-agent root cause analysis system
- Advanced guardrails (loop detection, tool allowlists)
- Evaluation framework with metrics
- Failure modes and recovery
- Production deployment patterns

### Homework (Optional)

1. Add a 4th agent for incident summarization
2. Implement custom tool for querying metrics
3. Add conditional routing based on severity
4. Experiment with different similarity thresholds

See you tomorrow! 🚀